# 05 — Plastic Waste Analysis: The Recycling Myth After National Sword

**Question:** What happened to U.S. plastic waste exports after China's National Sword policy?

**Note on data:** Weight data for plastic (HS 3915) has significant gaps in the Comtrade dataset — several World totals report zero. We use **FOB value (USD)** as the primary metric for this analysis, which is fully reported across all years.

**Sections:**
1. China's share of U.S. plastic exports by value
2. Where did plastic go? Destination-level pull
3. Comparison: Paper vs. Plastic post-ban trajectories


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import requests
import time

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

FIG_DIR = os.path.expanduser('~/recycling-analysis/figures')
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_csv('../data/raw/comtrade_raw.csv')
plastic = df[df['cmd_code'] == 3915].copy()

china   = plastic[plastic['partner_name'] == 'China'].copy()
world   = plastic[plastic['partner_name'] == 'World'].copy()

print('Plastic waste data (value-based — weight has gaps):')
merged = china.merge(world[['year','fob_value_usd','net_weight_mt']], on='year', suffixes=('_china','_world'))
merged['china_share_value'] = merged['fob_value_usd_china'] / merged['fob_value_usd_world']
merged['post'] = (merged['year'] >= 2018).astype(int)
print(merged[['year','fob_value_usd_china','fob_value_usd_world','china_share_value']].to_string())

## Chart 1: China's Share of U.S. Plastic Waste Exports (by value)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Absolute value
ax = axes[0]
w = world.sort_values('year')
c = china.sort_values('year')

ax.fill_between(w['year'], w['fob_value_usd'] / 1e6,
                alpha=0.2, color='gray', label='Rest of World')
ax.fill_between(c['year'], c['fob_value_usd'] / 1e6,
                alpha=0.7, color='#ff7f0e', label='China')
ax.axvline(x=2018, color='black', linestyle='--', linewidth=1, alpha=0.7)
ax.text(2018.1, w['fob_value_usd'].max() / 1e6 * 0.9, 'National Sword', fontsize=9)
ax.set_title('U.S. Plastic Waste Exports\n(China vs. Total World)', fontweight='bold')
ax.set_ylabel('Export Value (USD Millions)')
ax.set_xlabel('Year')
ax.set_xticks(range(2015, 2024))
ax.legend()

# Right: China share
ax2 = axes[1]
m = merged.sort_values('year')
ax2.plot(m['year'], m['china_share_value'] * 100,
         color='#ff7f0e', linewidth=2.5, marker='o', markersize=5)
ax2.axvline(x=2018, color='black', linestyle='--', linewidth=1, alpha=0.7)
ax2.text(2018.1, m['china_share_value'].max() * 100 * 0.9, 'National Sword', fontsize=9)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.set_title("China's Share of U.S. Plastic\nWaste Exports (by value)", fontweight='bold')
ax2.set_ylabel("China's Share")
ax2.set_xlabel('Year')
ax2.set_xticks(range(2015, 2024))

plt.suptitle('National Sword Essentially Ended U.S. Plastic Exports to China',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/05_plastic_china_share.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

## Pull Destination-Level Plastic Data

In [ ]:
API_KEY = os.environ.get('COMTRADE_KEY', '')

ALT_PARTNERS = {
    356: 'India',
    704: 'Vietnam',
    458: 'Malaysia',
    360: 'Indonesia',
    484: 'Mexico',
    410: 'South Korea',
    764: 'Thailand',
    608: 'Philippines',
    792: 'Turkey',
}

def fetch_plastic_data(partner_code, partner_name, years=range(2015, 2024)):
    rows = []
    for year in years:
        url = 'https://comtradeapi.un.org/data/v1/get/C/A/HS'
        params = {
            'reporterCode': '842',
            'partnerCode': str(partner_code),
            'cmdCode': '3915',              # Plastic waste
            'flowCode': 'X',
            'period': str(year),
            'subscription-key': API_KEY
        }
        try:
            r = requests.get(url, params=params, timeout=15)
            data = r.json()
            if data.get('data'):
                for item in data['data']:
                    rows.append({
                        'year': year,
                        'partner': partner_name,
                        'net_weight_mt': (item.get('netWgt') or 0) / 1000,
                        'fob_value_usd': item.get('primaryValue') or 0
                    })
        except Exception as e:
            print(f'  Error {partner_name} {year}: {e}')
        time.sleep(0.5)
    return rows

if API_KEY:
    all_rows = []
    for code, name in ALT_PARTNERS.items():
        print(f'Fetching {name}...')
        all_rows.extend(fetch_plastic_data(code, name))
    plastic_dest = pd.DataFrame(all_rows)
    plastic_dest.to_csv('../data/processed/plastic_destinations.csv', index=False)
    print(f'Saved {len(plastic_dest)} rows to plastic_destinations.csv')
    print(plastic_dest.groupby('partner')['fob_value_usd'].sum().sort_values(ascending=False))
else:
    print('No API key found. Set COMTRADE_KEY env variable.')

## Chart 2: Where Did U.S. Plastic Waste Go After National Sword?

In [ ]:
plastic_dest = pd.read_csv('../data/processed/plastic_destinations.csv')

# Add China for comparison (value-based)
china_plastic = df[(df['partner_name'] == 'China') & (df['cmd_code'] == 3915)][['year','fob_value_usd']].copy()
china_plastic['partner'] = 'China'

combined = pd.concat([
    plastic_dest[['year','partner','fob_value_usd']],
    china_plastic[['year','partner','fob_value_usd']]
])

pivot = combined.pivot_table(index='year', columns='partner', values='fob_value_usd', aggfunc='sum').fillna(0)
pivot = pivot.sort_index()

colors = {
    'China': '#d62728',
    'India': '#1f77b4',
    'Vietnam': '#2ca02c',
    'Malaysia': '#ff7f0e',
    'Indonesia': '#9467bd',
    'Mexico': '#8c564b',
    'South Korea': '#e377c2',
    'Thailand': '#7f7f7f',
    'Philippines': '#bcbd22',
    'Turkey': '#17becf',
}

fig, ax = plt.subplots(figsize=(12, 6))

for country in pivot.columns:
    lw = 3 if country == 'China' else 1.5
    ls = '-' if country == 'China' else '--'
    ax.plot(pivot.index, pivot[country] / 1e6,
            label=country, color=colors.get(country, 'gray'),
            linewidth=lw, linestyle=ls, marker='o', markersize=4)

ax.axvline(x=2018, color='black', linestyle=':', linewidth=1, alpha=0.7)
ax.text(2018.1, pivot.max().max() / 1e6 * 0.9, 'National Sword', fontsize=9)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Export Value (USD Millions)', fontsize=11)
ax.set_title('Where Did U.S. Plastic Waste Go After National Sword?\nExport Flows by Destination, 2015-2023',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9, framealpha=0.8)
ax.set_xticks(range(2015, 2024))

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/05_plastic_destinations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

## Chart 3: Paper vs. Plastic — How Different Were the Post-Ban Trajectories?

In [ ]:
# Load paper destinations for comparison
paper_dest = pd.read_csv('../data/processed/paper_destinations.csv')

# Total non-China flows per year for each commodity
paper_total = paper_dest.groupby('year')['fob_value_usd'].sum().reset_index()
paper_total.columns = ['year', 'paper_alt_value']

plastic_total = plastic_dest.groupby('year')['fob_value_usd'].sum().reset_index()
plastic_total.columns = ['year', 'plastic_alt_value']

# China flows
paper_china = df[(df['partner_name'] == 'China') & (df['cmd_code'] == 4707)][['year','fob_value_usd']].rename(columns={'fob_value_usd': 'paper_china_value'})
plastic_china = df[(df['partner_name'] == 'China') & (df['cmd_code'] == 3915)][['year','fob_value_usd']].rename(columns={'fob_value_usd': 'plastic_china_value'})

comp = paper_total.merge(plastic_total, on='year').merge(paper_china, on='year').merge(plastic_china, on='year')
comp = comp.sort_values('year')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Paper panel
ax = axes[0]
ax.stackplot(comp['year'],
             comp['paper_china_value'] / 1e6,
             comp['paper_alt_value'] / 1e6,
             labels=['China', 'Alt. Destinations'],
             colors=['#d62728', '#2ca02c'], alpha=0.7)
ax.axvline(x=2018, color='black', linestyle='--', linewidth=1)
ax.set_title('Waste Paper\nChina vs. Alt. Destinations', fontweight='bold')
ax.set_ylabel('Export Value (USD Millions)')
ax.set_xlabel('Year')
ax.set_xticks(range(2015, 2024))
ax.legend(loc='upper right', fontsize=9)

# Plastic panel
ax2 = axes[1]
ax2.stackplot(comp['year'],
              comp['plastic_china_value'] / 1e6,
              comp['plastic_alt_value'] / 1e6,
              labels=['China', 'Alt. Destinations'],
              colors=['#d62728', '#ff7f0e'], alpha=0.7)
ax2.axvline(x=2018, color='black', linestyle='--', linewidth=1)
ax2.set_title('Plastic Waste\nChina vs. Alt. Destinations', fontweight='bold')
ax2.set_ylabel('Export Value (USD Millions)')
ax2.set_xlabel('Year')
ax2.set_xticks(range(2015, 2024))
ax2.legend(loc='upper right', fontsize=9)

plt.suptitle('Paper Found New Markets. Plastic Mostly Did Not.',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/05_paper_vs_plastic_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

## Summary: Key Findings

| Finding | Detail |
|---|---|
| China share collapse | Plastic exports to China fell from ~38% to <1% of total value |
| No clear redirect | Unlike paper (Vietnam/Thailand), plastic had no dominant replacement market |
| The recycling myth | Much of U.S. plastic labeled 'recycled' was simply exported — when China stopped accepting it, the system broke |
| Paper vs plastic | Paper found new markets in SE Asia; plastic largely did not — a key distinction |
